# Paso 1 — De noticia a dato: extracción de eventos de extorsión con un LLM local

**Objetivo:** convertir el texto de una noticia en una fila estructurada (fecha, distrito, tipo de negocio, modalidad…) con un **LLM local** (Ollama) en vez de la API de Claude.



```
noticia (texto) ──► LLM local + esquema ──► JSON ──► Pydantic valida ──► fila en pandas
```

## 0. Preparación (solo la primera vez)

1. Instalar **Ollama** desde https://ollama.com (tiene instalador para Windows).
2. En una terminal, descargar un modelo. 
   ```
   ollama pull qwen2.5:7b
   ```
3. Instalar las librerías de Python:
   ```
   pip install ollama pydantic pandas
   ```

In [5]:
import json
from typing import Literal, Optional

import pandas as pd
from pydantic import BaseModel, Field, ValidationError, field_validator

MODELO = "qwen2.5:7b"  

## 1. El esquema: qué campos queremos sacar de cada noticia


Decisiones de diseño:
- **`Literal[...]`** = lista cerrada de categorías. Así el LLM no inventa 20 formas de decir "bodega" y luego se puede agrupar con pandas sin limpiar texto.
- **`Optional[...]`** = el campo puede quedar vacío (`null`) si la noticia no lo dice.
- **Sin nombres de personas** (víctimas ni sospechosos): cumplir con la Ley 29733 de datos personales.
- `tipo_negocio_afectado` es **el campo clave** para el cruce crimen–negocios que se pide

In [2]:
TipoNegocio = Literal[
    "bodega", "transporte_publico", "mototaxi", "restaurante", "mercado_o_puesto",
    "construccion", "colegio", "taller", "discoteca_o_bar", "otro", "no_especificado",
]
Modalidad = Literal[
    "cobro_de_cupo", "prestamo_gota_a_gota", "chalequeo", "amenaza_con_explosivo",
    "otro", "no_especificado",
]
MedioAmenaza = Literal[
    "whatsapp", "llamada", "nota_escrita", "ataque_al_local", "presencial",
    "redes_sociales", "otro", "no_especificado",
]


class Evento(BaseModel):
    """Un caso de extorsión extraído de una noticia."""

    es_extorsion: bool = Field(description="true solo si la noticia trata de un caso de extorsión")
    fecha_hecho: Optional[str] = Field(None, description="Fecha del hecho en formato YYYY-MM-DD; null si la noticia no la da")
    distrito: Optional[str] = Field(None, description="Distrito donde ocurrió, p. ej. 'San Juan de Lurigancho'")
    direccion_referencia: Optional[str] = Field(None, description="Avenida, calle, mercado o referencia del lugar del negocio")
    tipo_negocio_afectado: TipoNegocio
    modalidad: Modalidad
    medio_amenaza: MedioAmenaza
    hubo_ataque: bool = Field(description="true si hubo disparos, explosivo u otro ataque físico")
    monto_exigido_soles: Optional[float] = Field(None, description="Monto exigido en soles; null si no se menciona")
    organizacion_mencionada: Optional[str] = Field(None, description="Nombre de la banda u organización criminal, si se menciona")
    hubo_detenidos: Optional[bool] = Field(None, description="true/false si la noticia lo dice; null si no")
    resumen: str = Field(description="Una sola frase que resume el hecho, SIN nombres de personas")

    @field_validator("fecha_hecho")
    @classmethod
    def fecha_iso(cls, v):
        """Si hay fecha, debe ser YYYY-MM-DD válida."""
        if v is None:
            return v
        from datetime import date
        date.fromisoformat(v)  # lanza error si el formato es incorrecto
        return v


# Así ve el LLM el "contrato" (JSON Schema generado automáticamente):
print(json.dumps(Evento.model_json_schema(), indent=2, ensure_ascii=False)[:1200], "...")

{
  "description": "Un caso de extorsión extraído de una noticia.",
  "properties": {
    "es_extorsion": {
      "description": "true solo si la noticia trata de un caso de extorsión",
      "title": "Es Extorsion",
      "type": "boolean"
    },
    "fecha_hecho": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "description": "Fecha del hecho en formato YYYY-MM-DD; null si la noticia no la da",
      "title": "Fecha Hecho"
    },
    "distrito": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "description": "Distrito donde ocurrió, p. ej. 'San Juan de Lurigancho'",
      "title": "Distrito"
    },
    "direccion_referencia": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "descriptio

## 2. Pydantic en acción (sin LLM todavía)

Antes de usar el modelo, mirar qué hace Pydantic: **acepta** un JSON correcto y **rechaza** uno con errores.
Esto es lo que nos protege de que el LLM "invente" categorías o formatos.

In [3]:
json_bueno = """{
  "es_extorsion": true, "fecha_hecho": "2026-09-03", "distrito": "Comas",
  "direccion_referencia": "Av. Túpac Amaru", "tipo_negocio_afectado": "bodega",
  "modalidad": "cobro_de_cupo", "medio_amenaza": "whatsapp", "hubo_ataque": false,
  "monto_exigido_soles": 500, "organizacion_mencionada": null, "hubo_detenidos": null,
  "resumen": "Dueño de bodega recibe mensajes exigiendo cupo semanal."
}"""
print("✔ válido:", Evento.model_validate_json(json_bueno).tipo_negocio_afectado)

json_malo = json_bueno.replace('"bodega"', '"tiendita de abarrotes"').replace("2026-09-03", "3 de setiembre")
try:
    Evento.model_validate_json(json_malo)
except ValidationError as e:
    print(f"✘ rechazado con {e.error_count()} errores:")
    for err in e.errors():
        print("   -", err["loc"], "→", err["msg"][:80])

✔ válido: bodega
✘ rechazado con 2 errores:
   - ('fecha_hecho',) → Value error, Invalid isoformat string: '3 de setiembre'
   - ('tipo_negocio_afectado',) → Input should be 'bodega', 'transporte_publico', 'mototaxi', 'restaurante', 'merc


## 3. El prompt de extracción

El prompt le explica al modelo su rol y las reglas. El esquema (formato) se lo pasamos aparte,
así que aquí nos enfocamos en el criterio: qué cuenta como extorsión, qué hacer si falta un dato, y no poner nombres.

In [4]:
PROMPT_SISTEMA = """Eres un analista que convierte noticias policiales peruanas en datos estructurados.
Lee la noticia y completa los campos del esquema JSON.

Reglas:
- es_extorsion = true solo si hay exigencia de dinero bajo amenaza (cupos, gota a gota, chalequeo, etc.).
  Si la noticia es de otro delito (robo, hurto, homicidio sin extorsión), pon es_extorsion = false
  y usa "no_especificado" en las categorías.
- No inventes datos. Si la noticia no menciona algo, usa null o "no_especificado".
- La fecha va en formato YYYY-MM-DD. Si solo dice "ayer" o "el lunes", usa la fecha de publicación como referencia.
- NUNCA escribas nombres de personas (víctimas, sospechosos ni testigos). Sí puedes escribir nombres de bandas.
- Responde solo con el JSON."""

## 4. Llamar al LLM local y validar

`format=Evento.model_json_schema()` activa la salida estructurada de Ollama: el modelo queda obligado a responder
con JSON que sigue el esquema. Aun así validamos con Pydantic (doble seguro) y reintentamos si falla.

`temperature = 0` hace que el modelo sea lo más determinista posible: para extraer datos no queremos creatividad.

In [6]:
import ollama


def llamar_llm(noticia: str, fecha_publicacion: str, modelo: str = MODELO) -> str:
    """Envía la noticia al LLM local y devuelve el texto JSON de la respuesta."""
    respuesta = ollama.chat(
        model=modelo,
        messages=[
            {"role": "system", "content": PROMPT_SISTEMA},
            {"role": "user", "content": f"Fecha de publicación: {fecha_publicacion}\n\nNoticia:\n{noticia}"},
        ],
        format=Evento.model_json_schema(),
        options={"temperature": 0},
    )
    return respuesta.message.content


def extraer_evento(noticia: str, fecha_publicacion: str, intentos: int = 2) -> Optional[Evento]:
    """Llama al LLM y valida con Pydantic; reintenta si la respuesta no cumple el esquema."""
    for i in range(1, intentos + 1):
        texto = llamar_llm(noticia, fecha_publicacion)
        try:
            return Evento.model_validate_json(texto)
        except ValidationError as e:
            print(f"  intento {i}: respuesta inválida ({e.error_count()} errores), reintentando…")
    return None

## 5. Noticias de prueba

Tres noticias inventadas para practicar. La tercera fue a propósito un robo (no extorsión) para
comprobar que el modelo sabe filtrar.

Cuando funcione, reemplazar por noticias reales (copia y pega el texto de reportes periodísticos).
Más adelante, esta lista vendrá del scraper del módulo 1.

In [7]:
noticias = [
    {
        "id_noticia": "demo-001",
        "medio": "ejemplo",
        "fecha_publicacion": "2026-10-10",
        "texto": (
            "Una empresa de transporte que cubre la ruta entre San Juan de Lurigancho y el Cercado "
            "denunció que desde hace un mes recibe mensajes de WhatsApp en los que una banda que se hace "
            "llamar 'Los Malditos del Cerro' exige 15 soles diarios por cada unidad. El miércoles, dos "
            "sujetos en moto dispararon contra un bus estacionado en la avenida Próceres de la Independencia. "
            "No hubo heridos. La policía aún no reporta capturas."
        ),
    },
    {
        "id_noticia": "demo-002",
        "medio": "ejemplo",
        "fecha_publicacion": "2026-10-12",
        "texto": (
            "En Comas, el propietario de una bodega ubicada cerca del mercado Collique encontró ayer un "
            "artefacto explosivo en la puerta de su local junto a una nota escrita a mano que le exigía "
            "5 000 soles. Agentes de la Udex detonaron el explosivo de manera controlada. Vecinos señalan "
            "que otros negocios de la zona también reciben amenazas."
        ),
    },
    {
        "id_noticia": "demo-003",
        "medio": "ejemplo",
        "fecha_publicacion": "2026-10-13",
        "texto": (
            "Dos delincuentes asaltaron a una joven en un paradero de la avenida Arequipa, en Lince, y le "
            "robaron su celular. Serenos de la municipalidad lograron capturar a uno de los sujetos."
        ),
    },
]
print(len(noticias), "noticias cargadas")

3 noticias cargadas


## 6. Procesar todas las noticias y armar la tabla

Cada fila guarda de dónde salió el dato (`id_noticia`, `medio`, `fecha_publicacion`) más los campos extraídos.
Esto es la "trazabilidad": si un dato se ve raro, puedes volver a la noticia original.

In [8]:
filas = []
for n in noticias:
    print(f"Procesando {n['id_noticia']}…")
    evento = extraer_evento(n["texto"], n["fecha_publicacion"])
    if evento is None:
        print("  ✘ no se pudo extraer; queda para revisión manual")
        continue
    filas.append({
        "id_noticia": n["id_noticia"],
        "medio": n["medio"],
        "fecha_publicacion": n["fecha_publicacion"],
        **evento.model_dump(),
    })

df = pd.DataFrame(filas)
df

Procesando demo-001…
Procesando demo-002…
Procesando demo-003…


,id_noticia,medio,fecha_publicacion,es_extorsion,fecha_hecho,distrito,direccion_referencia,tipo_negocio_afectado,modalidad,medio_amenaza,hubo_ataque,monto_exigido_soles,organizacion_mencionada,hubo_detenidos,resumen
0,demo-001,ejemplo,2026-10-10,True,2026-10-09,"San Juan de Lurigancho, Cercado",avenida Próceres de la Independencia,transporte_publico,chalequeo,whatsapp,True,15.0,None,False,Una empresa de transporte que cubre la ruta en...
1,demo-002,ejemplo,2026-10-12,True,2026-10-11,Comas,NaN,bodega,amenaza_con_explosivo,nota_escrita,True,5000.0,None,None,El propietario de una bodega en Comas encontró...
2,demo-003,ejemplo,2026-10-13,False,2026-10-13,Lince,paradero de la avenida Arequipa,no_especificado,amenaza_con_explosivo,no_especificado,False,NaN,None,True,Dos delincuentes asaltaron a una joven en un p...


## 7. Filtrar, revisar y guardar

Solo quedan los casos de extorsión. El CSV resultante tiene las mismas columnas que tendrá la tabla `eventos` en Supabase.

In [9]:
extorsiones = df[df["es_extorsion"]].copy()
print(f"{len(extorsiones)} de {len(df)} noticias son extorsión\n")

print("Por tipo de negocio afectado:")
print(extorsiones["tipo_negocio_afectado"].value_counts(), "\n")
print("Por distrito:")
print(extorsiones["distrito"].value_counts())

extorsiones.to_csv("eventos_piloto.csv", index=False, encoding="utf-8-sig")
print("\nGuardado en eventos_piloto.csv")

2 de 3 noticias son extorsión

Por tipo de negocio afectado:
tipo_negocio_afectado
transporte_publico    1
bodega                1
Name: count, dtype: int64 

Por distrito:
distrito
San Juan de Lurigancho, Cercado    1
Comas                              1
Name: count, dtype: int64

Guardado en eventos_piloto.csv
